## FooDB JSON to MySQL Dump Conversion Notebook

This notebook provides a solution to convert a directory of FooDB JSON files into a single, normalized MySQL dump (`.sql`) file. It handles data parsing, schema inference, data cleaning, and generates optimized `CREATE TABLE` and `INSERT` statements.


In [ ]:
# Import necessary libraries
import os
import json
import pandas as pd
from tqdm.notebook import tqdm # Using tqdm.notebook for Colab progress bars

# Define the base directory where your JSON files are located
# This path is based on the previously loaded Nutrient.json file.
base_json_directory = '/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca'

# Define the output SQL dump file path
output_sql_file = '/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/foodb_generated.sql'

print(f"JSON files will be processed from: {base_json_directory}")
print(f"MySQL dump will be saved to: {output_sql_file}")


JSON files will be processed from: /content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca
MySQL dump will be saved to: /content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/foodb_generated.sql


### 1. Data Parsing & Inspection: Traversing JSON Files

First, we need to locate all JSON files within the specified base directory and its subdirectories. We'll use `os.walk` for recursive traversal and filter for `.json` files.

In [ ]:
def find_all_json_files(base_dir):
    """Recursively finds all .json files in a given directory."""
    json_files = []
    for root, _, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.json'):
                json_files.append(os.path.join(root, file))
    return json_files

# Get a list of all JSON files
all_json_files = find_all_json_files(base_json_directory)
print(f"Found {len(all_json_files)} JSON files.")

# Display the first few for inspection
print("First 5 JSON files found:")
for f in all_json_files[:5]:
    print(f)

Found 29 JSON files.
First 5 JSON files found:
/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca/Compound.json
/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca/Content.json
/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca/AccessionNumber.json
/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca/CompoundAlternateParent.json
/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca/CompoundExternalDescriptor.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/www.foodb.ca/Nutrient.json'
df = pd.read_json(file_path, lines=True)
display(df.head())

,id,legacy_id,public_id,name,export,state,annotation_quality,description,wikipedia_id,comments,...,eafus_id,dfc_name,compound_source,metabolism,synthesis_citations,general_citations,creator_id,updater_id,created_at,updated_at
0,1,10930.0,FDBN00001,Fat,False,NaN,low,None,None,NaN,...,NaN,NaN,DUKE,NaN,NaN,NaN,NaN,NaN,2014-11-05 13:42:10+00:00,2014-11-05 13:42:10+00:00
1,2,10946.0,FDBN00002,Proteins,False,NaN,low,None,None,NaN,...,NaN,NaN,DUKE,NaN,NaN,NaN,NaN,NaN,2014-11-05 13:42:15+00:00,2014-11-05 13:42:15+00:00
2,3,16037.0,FDBN00003,Carbohydrate,False,NaN,low,Carbohydrates (or saccharides) are organic com...,Carbohydrate,NaN,...,NaN,NaN,DUKE,NaN,NaN,NaN,NaN,NaN,2014-11-05 13:44:06+00:00,2014-11-05 13:44:06+00:00
3,4,23404.0,FDBN00004,Fatty acids,False,NaN,low,None,None,NaN,...,1335.0,NaN,EAFUS,NaN,NaN,NaN,NaN,NaN,2014-11-05 13:46:00+00:00,2014-11-05 13:46:00+00:00
4,5,11134.0,FDBN00005,Fiber (dietary),False,NaN,low,None,None,NaN,...,NaN,NaN,DUKE,NaN,NaN,NaN,NaN,NaN,2014-11-05 13:47:36+00:00,2014-11-05 13:47:36+00:00


### 2. Dynamic Schema Discovery

This section focuses on automatically inferring table schemas (column names and data types) from the JSON files. We'll iterate through the files, parse their content, and build a consolidated schema definition. This will also involve identifying potential primary/foreign key relationships and nested structures.

In [ ]:
import collections
from tqdm.notebook import tqdm

# Helper to map Python types to MySQL types
def get_mysql_type(value):
    if isinstance(value, bool):
        return 'BOOLEAN'
    elif isinstance(value, int):
        # Check for potential large integers (e.g., IDs) or small integers
        if value > 2147483647 or value < -2147483648: # Max/min for signed INT
            return 'BIGINT'
        else:
            return 'INT'
    elif isinstance(value, float):
        return 'DOUBLE'
    elif isinstance(value, str):
        # Basic check for datetime-like strings
        if len(value) > 10 and (('T' in value and '-' in value) or (':' in value and '-' in value)):
            return 'DATETIME'
        # Basic check for potentially large text fields (e.g., descriptions)
        if len(value) > 255: # A common limit for VARCHAR
            return 'TEXT'
        return 'VARCHAR(255)'
    elif value is None:
        return 'NULLABLE'
    else:
        return 'TEXT' # Default for complex types or unknown

def infer_schema_from_json(json_data, current_schema=None, parent_key=""):
    """Recursively infers schema from a single JSON object."""
    if current_schema is None:
        current_schema = collections.defaultdict(lambda: {'type': 'NULLABLE', 'nullable': True})

    if isinstance(json_data, dict):
        for key, value in json_data.items():
            full_key = f"{parent_key}_{key}" if parent_key else key
            if isinstance(value, (dict, list)):
                # For nested objects or arrays, we will handle them as separate tables later
                # For now, just mark their presence if they are direct children
                if not parent_key: # Only process top-level nested structures here
                    current_schema[full_key]['is_nested'] = True
                    current_schema[full_key]['type'] = 'JSON' # Mark as JSON type for initial pass
                # Recursive call for nested dictionaries, but we won't flatten them in place for schema
                infer_schema_from_json(value, current_schema, parent_key=full_key)
            else:
                inferred_type = get_mysql_type(value)
                current_type = current_schema[full_key]['type']

                if current_type == 'NULLABLE':
                    current_schema[full_key]['type'] = inferred_type
                elif inferred_type == 'NULLABLE':
                    current_schema[full_key]['nullable'] = True
                else:
                    # Promote type if necessary (e.g., INT to BIGINT, VARCHAR to TEXT)
                    if current_type == 'INT' and inferred_type == 'BIGINT':
                        current_schema[full_key]['type'] = 'BIGINT'
                    elif current_type == 'VARCHAR(255)' and inferred_type == 'TEXT':
                        current_schema[full_key]['type'] = 'TEXT'
                    elif current_type != inferred_type and inferred_type != 'VARCHAR(255)': # Be more general if types mismatch but not string
                        # For simplicity, if types conflict and it's not a simple string upgrade, use TEXT
                        if current_type != 'TEXT' and inferred_type != 'NULLABLE':
                            current_schema[full_key]['type'] = 'TEXT'

    elif isinstance(json_data, list):
        # For lists, we infer schema from the first element if it's an object
        if json_data and isinstance(json_data[0], dict):
            infer_schema_from_json(json_data[0], current_schema, parent_key=parent_key)

    return current_schema


# Dictionary to store schemas for each 'table'
table_schemas = {}

print("Inferring schemas from JSON files...")
for file_path in tqdm(all_json_files, desc="Inferring Schemas"):
    try:
        # Determine a 'table name' from the file name
        file_name = os.path.basename(file_path)
        table_name = file_name.replace('.json', '').lower()

        # Read only a sample to avoid memory issues for very large files
        # For files where each line is a JSON object (common in FooDB)
        sample_data = []
        with open(file_path, 'r') as f:
            for i, line in enumerate(f):
                try:
                    sample_data.append(json.loads(line))
                    if i >= 100: # Sample first 100 lines for schema inference
                        break
                except json.JSONDecodeError:
                    continue # Skip malformed lines

        if not sample_data:
            # Handle empty files or files that don't contain line-delimited JSON
            # Try reading as a single JSON object if lines=True failed
            try:
                with open(file_path, 'r') as f:
                    full_content = json.load(f)
                    if isinstance(full_content, list):
                        if full_content and isinstance(full_content[0], dict):
                            sample_data = full_content[:100]
                    elif isinstance(full_content, dict):
                        sample_data = [full_content]
            except (json.JSONDecodeError, UnicodeDecodeError):
                # If it's not line-delimited and not a single JSON object, skip
                tqdm.write(f"Skipping {file_name}: Not a valid JSON format or empty.")
                continue

        if table_name not in table_schemas:
            table_schemas[table_name] = {}

        for record in sample_data:
            inferred = infer_schema_from_json(record)
            # Merge inferred schema with existing one for the table
            for col, details in inferred.items():
                if col not in table_schemas[table_name]:
                    table_schemas[table_name][col] = details
                else:
                    # Merge types if necessary (e.g., if one sample had INT, another had BIGINT, take BIGINT)
                    existing_type = table_schemas[table_name][col]['type']
                    new_type = details['type']

                    if new_type != 'NULLABLE' and (existing_type == 'NULLABLE' or existing_type == new_type):
                        table_schemas[table_name][col]['type'] = new_type
                    elif existing_type == 'INT' and new_type == 'BIGINT':
                        table_schemas[table_name][col]['type'] = 'BIGINT'
                    elif existing_type == 'VARCHAR(255)' and new_type == 'TEXT':
                        table_schemas[table_name][col]['type'] = 'TEXT'
                    # If both are valid types and different, take the more general one (e.g., INT -> DOUBLE)
                    elif existing_type != new_type and new_type != 'NULLABLE':
                        # A more robust type promotion logic could go here
                        if existing_type == 'INT' and new_type == 'DOUBLE':
                            table_schemas[table_name][col]['type'] = 'DOUBLE'
                        elif existing_type == 'BOOLEAN' and new_type == 'INT':
                            table_schemas[table_name][col]['type'] = 'INT'
                        elif existing_type != 'TEXT' and new_type != 'TEXT': # Avoid overwriting TEXT with less general types
                             table_schemas[table_name][col]['type'] = 'TEXT' # Fallback to TEXT for significant mismatches

                    # Update nullability
                    table_schemas[table_name][col]['nullable'] = table_schemas[table_name][col]['nullable'] and details.get('nullable', True)
                    # Update is_nested property
                    if details.get('is_nested'):
                        table_schemas[table_name][col]['is_nested'] = True

    except Exception as e:
        tqdm.write(f"Error processing {file_name}: {e}")
        continue

print("Schema inference complete. Displaying inferred schemas:")
for table_name, schema in table_schemas.items():
    print(f"\nTable: {table_name}")
    for col, details in schema.items():
        print(f"  {col}: {details['type']} (Nullable: {details['nullable']}) {'(Nested)' if details.get('is_nested') else ''}")


Inferring schemas from JSON files...


Inferring Schemas:   0%|          | 0/29 [00:00<?, ?it/s]

Skipping EnzymeSynonym.json: Not a valid JSON format or empty.
Skipping PfamMembership.json: Not a valid JSON format or empty.
Skipping PdbIdentifier.json: Not a valid JSON format or empty.
Skipping MapItemsPathway.json: Not a valid JSON format or empty.
Skipping Pfam.json: Not a valid JSON format or empty.
Skipping Sequence.json: Not a valid JSON format or empty.
Schema inference complete. Displaying inferred schemas:

Table: compound
  id: INT (Nullable: True) 
  public_id: VARCHAR(255) (Nullable: True) 
  name: TEXT (Nullable: True) 
  state: VARCHAR(255) (Nullable: True) 
  annotation_quality: VARCHAR(255) (Nullable: True) 
  description: TEXT (Nullable: True) 
  cas_number: VARCHAR(255) (Nullable: True) 
  moldb_smiles: TEXT (Nullable: True) 
  moldb_inchi: TEXT (Nullable: True) 
  moldb_mono_mass: VARCHAR(255) (Nullable: True) 
  moldb_inchikey: TEXT (Nullable: True) 
  moldb_iupac: TEXT (Nullable: True) 
  kingdom: VARCHAR(255) (Nullable: True) 
  superklass: VARCHAR(255) (Nulla

### 3. Data Cleaning & Normalization

This step involves processing the JSON data based on the inferred schemas. We will flatten nested objects and arrays into separate tables and sanitize data for SQL insertion. Each top-level JSON file will correspond to a main table, and nested structures (identified by `is_nested: True` in the schema) will become new, related tables.

In [ ]:
import re

# Dictionary to hold processed data for each table
processed_data = collections.defaultdict(list)

# Helper function to sanitize string values for SQL
def sanitize_sql_value(value):
    if value is None:
        return 'NULL'
    if isinstance(value, bool):
        return 'TRUE' if value else 'FALSE'
    if isinstance(value, (int, float)):
        return str(value)

    # Convert datetime objects to string if they somehow sneaked in
    if isinstance(value, pd.Timestamp):
        return f"'{value.strftime('%Y-%m-%d %H:%M:%S')}'"

    # For strings, escape single quotes and handle backslashes
    # Also, remove null bytes which can cause issues with MySQL
    s_value = str(value).replace("'", "''").replace('\\', '\\\\').replace('\x00', '')
    return f"'{s_value}'"

def process_json_record(table_name, record, schemas, parent_id=None, parent_table_name=None):
    """Processes a single JSON record, extracting main table data and nested data."""
    main_table_record = {}
    nested_records = collections.defaultdict(list)

    # Generate a unique ID for the main record if 'id' is not present or is None
    # For simplicity, we'll auto-increment here. In a real DB, this would be auto_increment.
    record_id = record.get('id')
    if record_id is None or (isinstance(record_id, str) and not record_id.strip()):
        # This needs a more robust global ID generation if 'id' is not reliable
        # For now, we'll assign a temporary sequential ID during processing if missing
        record_id = len(processed_data[table_name]) + 1 # Simple sequential ID
        record['id'] = record_id # Add to the record for potential nested lookups

    # Ensure ID is an integer for consistency if it was a string representation
    try:
        record_id = int(record_id)
    except (ValueError, TypeError):
        record_id = len(processed_data[table_name]) + 1 # Fallback to new ID if conversion fails
        record['id'] = record_id

    current_table_schema = schemas.get(table_name, {})

    for col_name, col_schema in current_table_schema.items():
        original_key = col_name.split('_')[-1] if '_' in col_name and not col_schema.get('is_nested') else col_name
        value = record.get(original_key)

        if col_schema.get('is_nested') and original_key in record:
            nested_data = record[original_key]
            nested_table_name = col_name # Use the full_key as nested table name

            if isinstance(nested_data, list):
                for item in nested_data:
                    if isinstance(item, dict):
                        # Pass parent_id and parent_table_name to link nested records
                        item_id = item.get('id')
                        if item_id is None or (isinstance(item_id, str) and not item_id.strip()):
                             item_id = len(processed_data[nested_table_name]) + 1
                             item['id'] = item_id

                        processed_nested_item = process_json_record(
                            nested_table_name, item, schemas,
                            parent_id=record_id, parent_table_name=table_name
                        )
                        nested_records[nested_table_name].append(processed_nested_item)
            elif isinstance(nested_data, dict):
                # A single nested object will be treated as one record in the nested table
                nested_records[nested_table_name].append(process_json_record(
                    nested_table_name, nested_data, schemas,
                    parent_id=record_id, parent_table_name=table_name
                ))
        else:
            main_table_record[col_name] = value

    # Add parent foreign key if this is a nested record being processed recursively
    if parent_id is not None and parent_table_name is not None:
        fk_column_name = f"{parent_table_name}_id"
        main_table_record[fk_column_name] = parent_id
        # Also update the schema for this new FK column if it doesn't exist
        if fk_column_name not in schemas.get(table_name, {}):
             schemas[table_name][fk_column_name] = {'type': 'INT', 'nullable': False}

    # Store the processed main record
    processed_data[table_name].append(main_table_record)

    # Store nested records separately
    for nested_table, recs in nested_records.items():
        processed_data[nested_table].extend(recs)

    return main_table_record # Return the processed main record for recursive calls

# Iterate through all JSON files again, this time processing data
print("Processing and normalizing JSON data...")
for file_path in tqdm(all_json_files, desc="Processing Data"):
    try:
        file_name = os.path.basename(file_path)
        table_name = file_name.replace('.json', '').lower()

        temp_data_for_table = []
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    record = json.loads(line)
                    # Process each record (handles nested structures internally)
                    process_json_record(table_name, record, table_schemas)
                except json.JSONDecodeError:
                    continue # Skip malformed lines

        # If it was not line-delimited, try reading as a single object
        if not temp_data_for_table and table_name not in processed_data:
            try:
                with open(file_path, 'r') as f:
                    full_content = json.load(f)
                    if isinstance(full_content, list):
                        for record in full_content:
                            if isinstance(record, dict):
                                process_json_record(table_name, record, table_schemas)
                    elif isinstance(full_content, dict):
                        process_json_record(table_name, full_content, table_schemas)
            except (json.JSONDecodeError, UnicodeDecodeError):
                # Already handled by schema inference skip
                pass

    except Exception as e:
        tqdm.write(f"Error processing data from {file_name}: {e}")
        continue

print("Data normalization complete. Sample of processed data:")
for table, data in list(processed_data.items())[:3]: # Display first 3 tables
    print(f"\nTable: {table}, {len(data)} rows")
    if data:
        print(data[0]) # Print first record
    else:
        print("No data.")


Processing and normalizing JSON data...


Processing Data:   0%|          | 0/29 [00:00<?, ?it/s]

Data normalization complete. Sample of processed data:

Table: compound, 70477 rows
{'id': 4, 'public_id': 4, 'name': "Cyanidin 3-(6''-acetyl-galactoside)", 'state': None, 'annotation_quality': None, 'description': "Constituent of the leaves of Nymphaea alba [CCD]. Cyanidin 3-(6''-acetyl-galactoside) is found in lowbush blueberry and highbush blueberry.", 'cas_number': None, 'moldb_smiles': None, 'moldb_inchi': None, 'moldb_mono_mass': None, 'moldb_inchikey': None, 'moldb_iupac': None, 'kingdom': 'Organic compounds', 'superklass': 'Phenylpropanoids and polyketides', 'klass': 'Flavonoids', 'subklass': 'Flavonoid glycosides'}

Table: content, 5691011 rows
{'id': 1, 'source_id': 1, 'source_type': None, 'food_id': 1, 'orig_food_id': 1, 'orig_food_common_name': None, 'orig_food_scientific_name': None, 'orig_food_part': None, 'orig_source_id': 1, 'orig_source_name': None, 'orig_content': None, 'orig_min': None, 'orig_max': None, 'orig_unit': None, 'orig_citation': 'DUKE', 'citation': 'DUKE',

### 4. Generate SQLite Dump File

Now, we will generate the SQLite dump file (.sql). This involves two main parts:
1.  **Generating `CREATE TABLE` statements:** Based on the inferred schemas, we will create SQL `CREATE TABLE` statements, adjusting data types for SQLite compatibility and adding primary/foreign keys.
2.  **Generating `INSERT INTO` statements:** We will iterate through the processed data and generate `INSERT INTO` statements, batching them for performance and sanitizing all values.

In [ ]:
# Helper function to map inferred types to SQLite types
def get_sqlite_type(mysql_type):
    if mysql_type == 'INT':
        return 'INTEGER'
    elif mysql_type == 'BIGINT':
        return 'INTEGER'
    elif mysql_type == 'BOOLEAN':
        return 'BOOLEAN' # SQLite stores boolean as INTEGER (0 or 1)
    elif mysql_type == 'DOUBLE':
        return 'REAL'
    elif mysql_type.startswith('VARCHAR'):
        return 'TEXT'
    elif mysql_type == 'TEXT':
        return 'TEXT'
    elif mysql_type == 'DATETIME':
        return 'TEXT' # SQLite typically stores dates/times as TEXT, REAL or INTEGER
    elif mysql_type == 'NULLABLE': # Should not happen if schema inference is robust
        return 'TEXT'
    else:
        return 'TEXT'

def generate_create_table_statement(table_name, schema):
    columns_sql = []
    primary_key_columns = []

    # First pass to collect columns and identify potential primary keys
    for col_name, col_details in schema.items():
        sqlite_type = get_sqlite_type(col_details['type'])
        null_constraint = 'NOT NULL' if not col_details['nullable'] else 'NULL'

        # Identify potential primary keys (simple 'id' column)
        if col_name == 'id' and sqlite_type == 'INTEGER':
            primary_key_columns.append(col_name)
            # SQLite's INTEGER PRIMARY KEY is special: auto-incrementing if defined like this
            columns_sql.append(f"{col_name} {sqlite_type} PRIMARY KEY AUTOINCREMENT {null_constraint}")
        elif col_name.endswith('_id') and sqlite_type == 'INTEGER': # Potential Foreign Keys
             columns_sql.append(f"{col_name} {sqlite_type} {null_constraint}")
        else:
            columns_sql.append(f"{col_name} {sqlite_type} {null_constraint}")

    # For tables without an 'id' or if 'id' wasn't INTEGER PRIMARY KEY AUTOINCREMENT, add a generic one if needed
    if not primary_key_columns and not any('PRIMARY KEY' in col_sql for col_sql in columns_sql):
        # Add a default auto-incrementing primary key if none found and it's a 'main' table
        if not table_name.endswith('_id'): # Avoid adding PK to what might be a junction table or nested table FK
            columns_sql.insert(0, 'row_id INTEGER PRIMARY KEY AUTOINCREMENT')

    create_table_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n  " + ",\n  ".join(columns_sql) + "\n);"
    return create_table_sql

def generate_insert_statement(table_name, data_batch):
    if not data_batch:
        return ""

    # Ensure consistent column order for all records in the batch
    # Use the keys of the first record in the batch as the column order
    # This assumes all records in the batch have the same set of keys, which they should if processed correctly
    columns = list(data_batch[0].keys())

    # Filter out columns that are not in the schema (e.g., transient 'id' used during processing)
    # We rely on the schema for the final table structure
    schema_cols = list(table_schemas.get(table_name, {}).keys())
    insert_columns = [col for col in columns if col in schema_cols or (col == 'row_id' and 'row_id' in columns)]

    # If 'id' is defined as PRIMARY KEY AUTOINCREMENT, we should not include it in the INSERT statement
    # unless it's explicitly provided and needs to be set. For SQLite, it's often omitted.
    create_table_sql_snippet = generate_create_table_statement(table_name, table_schemas.get(table_name, {}))
    if 'id INTEGER PRIMARY KEY AUTOINCREMENT' in create_table_sql_snippet and 'id' in insert_columns:
        insert_columns.remove('id')
    elif 'row_id INTEGER PRIMARY KEY AUTOINCREMENT' in create_table_sql_snippet and 'row_id' in insert_columns:
        insert_columns.remove('row_id')

    cols_str = ", ".join(insert_columns)
    values_list = []

    for record in data_batch:
        # Ensure values align with the insert_columns order
        values = []
        for col in insert_columns:
            val = record.get(col)
            # Special handling for boolean values for SQLite (TRUE/FALSE -> 1/0)
            if isinstance(val, bool):
                values.append('1' if val else '0')
            else:
                values.append(sanitize_sql_value(val))
        values_list.append(f"({', '.join(values)})")

    # Use INSERT OR REPLACE INTO to handle potential ID conflicts, or just INSERT INTO if 'id' is auto-increment
    # For SQLite, it's common to use INSERT INTO and let AUTOINCREMENT handle IDs.
    insert_sql = f"INSERT INTO {table_name} ({cols_str}) VALUES {', '.join(values_list)};"
    return insert_sql


# Open the SQL dump file for writing
with open(output_sql_file, 'w') as f_sql:
    # SQLite Preamble
    f_sql.write("PRAGMA foreign_keys = OFF;\n")
    f_sql.write("BEGIN TRANSACTION;\n")

    # Generate and write CREATE TABLE statements
    print("\nGenerating CREATE TABLE statements...")
    for table_name, schema in tqdm(table_schemas.items(), desc="Creating Tables"):
        create_sql = generate_create_table_statement(table_name, schema)
        f_sql.write(create_sql + "\n\n")

    # Generate and write INSERT INTO statements
    print("\nGenerating INSERT INTO statements...")
    batch_size = 1000 # Optimal batch size for inserts
    total_rows_inserted = 0
    total_tables_processed = 0

    for table_name, records in tqdm(processed_data.items(), desc="Inserting Data"):
        if not records:
            continue # Skip empty tables

        total_tables_processed += 1
        for i in range(0, len(records), batch_size):
            batch = records[i:i + batch_size]
            if batch:
                insert_sql = generate_insert_statement(table_name, batch)
                f_sql.write(insert_sql + "\n")
                total_rows_inserted += len(batch)

    # SQLite Postamble
    f_sql.write("COMMIT;\n")
    f_sql.write("PRAGMA foreign_keys = ON;\n")

print(f"\nSQLite dump file generated successfully at: {output_sql_file}")
print(f"Total tables processed: {total_tables_processed}")
print(f"Total rows inserted: {total_rows_inserted}")



Generating CREATE TABLE statements...


Creating Tables:   0%|          | 0/23 [00:00<?, ?it/s]


Generating INSERT INTO statements...


Inserting Data:   0%|          | 0/23 [00:00<?, ?it/s]


SQLite dump file generated successfully at: /content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/foodb_generated.sql
Total tables processed: 23
Total rows inserted: 7845531


### 5. Generate SQLite Schema-Only Dump File

This section generates an SQLite dump file that contains only the `CREATE TABLE` statements, without any data. This is useful for quickly setting up an empty database with the correct structure.

In [ ]:
# Define the output SQL schema-only dump file path
output_schema_only_sql_file = '/content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/foodb_generated_schema_only.sql'

print(f"SQLite schema-only dump will be saved to: {output_schema_only_sql_file}")

with open(output_schema_only_sql_file, 'w') as f_sql_schema_only:
    # SQLite Preamble
    f_sql_schema_only.write("PRAGMA foreign_keys = OFF;\n")
    f_sql_schema_only.write("BEGIN TRANSACTION;\n")

    # Generate and write CREATE TABLE statements
    print("\nGenerating CREATE TABLE statements for schema-only dump...")
    for table_name, schema in tqdm(table_schemas.items(), desc="Creating Tables (Schema Only)"):
        create_sql = generate_create_table_statement(table_name, schema)
        f_sql_schema_only.write(create_sql + "\n\n")

    # SQLite Postamble
    f_sql_schema_only.write("COMMIT;\n")
    f_sql_schema_only.write("PRAGMA foreign_keys = ON;\n")

print(f"\nSQLite schema-only dump file generated successfully at: {output_schema_only_sql_file}")

SQLite schema-only dump will be saved to: /content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/foodb_generated_schema_only.sql

Generating CREATE TABLE statements for schema-only dump...


Creating Tables (Schema Only):   0%|          | 0/23 [00:00<?, ?it/s]


SQLite schema-only dump file generated successfully at: /content/drive/MyDrive/Hackathon/Build with Gemma : ESTI/foodb_generated_schema_only.sql
